In [63]:
import numpy as np
import pandas as pd
import random
import string
import mmh3
import math
import time

### Задание 1

Реализация Bloom-Filter на одной хеш-функции.

In [49]:
class BloomFilter:
    def __init__(self, capacity):
        self.capacity = capacity
        self.array_size = (capacity + 31) // 32
        self.array = np.zeros(self.array_size, dtype=np.uint32)
        self.bit_count = 0

    def _compute_hash(self, element):
        return mmh3.hash(element) % self.capacity

    def put(self, element):
        bit_index = self._compute_hash(element)
        array_index = bit_index // 32
        bit_offset = bit_index % 32
        if not (self.array[array_index] & (1 << bit_offset)):
            self.array[array_index] |= (1 << bit_offset)
            self.bit_count += 1

    def get(self, element):
        bit_index = self._compute_hash(element)
        array_index = bit_index // 32
        bit_offset = bit_index % 32
        return (self.array[array_index] & (1 << bit_offset)) != 0

    def size(self):
        return self.bit_count


def generate_random_string(length):
    characters = string.ascii_letters + string.digits
    random_string = ''.join(random.choice(characters) for _ in range(length))
    return random_string

In [3]:
def run_experiment(sizes_bf, sizes_set):
    bloom_filter = BloomFilter(sizes_bf)
    unique_strings = [generate_random_string(10) for _ in range(sizes_set)]
    false_positive_count = 0

    for s in unique_strings:
        if bloom_filter.get(s):
            false_positive_count += 1
        bloom_filter.put(s)

    ones_count = bloom_filter.size()
    return {
        'bf_size': sizes_bf,
        'set_size': sizes_set,
        'fp_count': false_positive_count,
        'ones_count': ones_count
    }

sizes_bf = [8, 64, 1024, 65536, 16777216]
sizes_set = [5, 50, 500, 5000, 5000000]

pd.DataFrame([run_experiment(bf_size, set_size) for bf_size in sizes_bf for set_size in sizes_set])

,bf_size,set_size,fp_count,ones_count
0,8,5,1,4
1,8,50,42,8
2,8,500,492,8
3,8,5000,4992,8
4,8,5000000,4999992,8
5,64,5,0,5
6,64,50,13,37
7,64,500,436,64
8,64,5000,4936,64
9,64,5000000,4999936,64


### Задание 2

Реализация Bloom Filter на k хеш-функций.

In [69]:
class BloomFilter:
    def __init__(self, capacity, k):
        self.capacity = capacity
        self.k = k
        self.array_size = (capacity + 31) // 32
        self.array = np.zeros(self.array_size, dtype=np.uint32)
        self.bit_count = 0

    def _compute_hashes(self, element):
        return [mmh3.hash(element, seed=i) % self.capacity for i in range(self.k)]

    def put(self, element):
        for bit_index in self._compute_hashes(element):
            array_index = bit_index // 32
            bit_offset = bit_index % 32
            if not (self.array[array_index] & (1 << bit_offset)):
                self.array[array_index] |= (1 << bit_offset)
                self.bit_count += 1

    def get(self, element):
        for bit_index in self._compute_hashes(element):
            array_index = bit_index // 32
            bit_offset = bit_index % 32
            if not (self.array[array_index] & (1 << bit_offset)):
                return False
        return True

    def size(self):
        return self.bit_count / self.k

In [70]:
def run_experiment(sizes_bf, sizes_set, k):
    bloom_filter = BloomFilter(sizes_bf, k)
    unique_strings = [generate_random_string(10) for _ in range(sizes_set)]
    false_positive_count = 0

    for s in unique_strings:
        if bloom_filter.get(s):
            false_positive_count += 1
        bloom_filter.put(s)

    ones_count = bloom_filter.size()
    size_estimation_error = abs(ones_count - sizes_set)
    
    return {
        'bf_size': sizes_bf,
        'set_size': sizes_set,
        'k': k,
        'fp_count': false_positive_count,
        'ones_count': ones_count,
        'size_estimation_error': size_estimation_error
    }


sizes_bf = [8, 64, 1024, 65536, 16777216]
sizes_set = [5, 50, 500, 5000, 5000000]
k_values = [1, 2, 3, 4]

df = pd.DataFrame([
    run_experiment(bf_size, set_size, k) 
    for bf_size in sizes_bf 
    for set_size in sizes_set 
    for k in k_values
])

In [73]:
df[:51]

,bf_size,set_size,k,fp_count,ones_count,size_estimation_error
0,8,5,1,2,3.000000,2.000000e+00
1,8,5,2,1,2.500000,2.500000e+00
2,8,5,3,1,2.333333,2.666667e+00
3,8,5,4,1,1.750000,3.250000e+00
4,8,50,1,42,8.000000,4.200000e+01
5,8,50,2,45,4.000000,4.600000e+01
6,8,50,3,45,2.666667,4.733333e+01
7,8,50,4,46,2.000000,4.800000e+01
8,8,500,1,492,8.000000,4.920000e+02
9,8,500,2,494,4.000000,4.960000e+02


In [74]:
df[51:]

,bf_size,set_size,k,fp_count,ones_count,size_estimation_error
51,1024,500,4,90,2.217500e+02,2.782500e+02
52,1024,5000,1,3986,1.014000e+03,3.986000e+03
53,1024,5000,2,4227,5.120000e+02,4.488000e+03
54,1024,5000,3,4386,3.413333e+02,4.658667e+03
55,1024,5000,4,4482,2.560000e+02,4.744000e+03
56,1024,5000000,1,4998976,1.024000e+03,4.998976e+06
57,1024,5000000,2,4999225,5.120000e+02,4.999488e+06
58,1024,5000000,3,4999373,3.413333e+02,4.999659e+06
59,1024,5000000,4,4999456,2.560000e+02,4.999744e+06
60,65536,5,1,0,5.000000e+00,0.000000e+00


### Задание 3

Реализация Counting Bloom Filter на k хеш-функций.

In [66]:
class CountingBloomFilter:
    def __init__(self, k, n, cap):
        self.filter_size = n
        self.filter = 0
        self.cap = cap
        self.num_of_funcs = k
        self.filled_mask = (1 << cap) - 1

    def _get_counter(self, index):
        shift = index * self.cap
        return (self.filter >> shift) & self.filled_mask

    def _set_counter(self, index, value):
        shift = index * self.cap
        self.filter &= ~(self.filled_mask << shift)
        self.filter |= (value & self.filled_mask) << shift

    def put(self, s):
        for i in range(self.num_of_funcs):
            index = mmh3.hash(s, i) % self.filter_size
            current_counter = self._get_counter(index)
            if current_counter + 1 < (1 << self.cap):
                self._set_counter(index, current_counter + 1)

    def get(self, s, theta):
        for i in range(self.num_of_funcs):
            index = mmh3.hash(s, i) % self.filter_size
            current_counter = self._get_counter(index)
            if current_counter < theta:
                return False
        return True

    def size(self):
        total = 0
        for i in range(self.filter_size):
            total += self._get_counter(i)
        return total / self.num_of_funcs

In [67]:
def run_experiment(n, set_size, k, cap, theta=1):
    cbf = CountingBloomFilter(k=k, n=n, cap=cap)
    unique_strings = [generate_random_string(10) for _ in range(set_size)]
    false_negative_count = 0
    total_count = 0

    for s in unique_strings:
        cbf.put(s)

    for s in unique_strings:
        if not cbf.get(s, theta=theta):
            false_negative_count += 1

    total_count = cbf.size()

    return {
        'n': n,
        'set_size': set_size,
        'k': k,
        'cap': cap,
        'theta': theta,
        'false_negative_count': false_negative_count,
        'average_count': total_count
    }


n = 2048
set_size = 1000
caps = [1, 2, 4, 8]
k_values = [1, 2, 3, 5]
theta = 1
results = []

for cap in caps:
    result = run_experiment(n=n, set_size=set_size, k=3, cap=cap, theta=theta)
    results.append(result)

for k in k_values:
    result = run_experiment(n=n, set_size=set_size, k=k, cap=4, theta=theta)
    results.append(result)

pd.DataFrame(results)

,n,set_size,k,cap,theta,false_negative_count,average_count
0,2048,1000,3,1,1,0,527.333333
1,2048,1000,3,2,1,0,936.000000
2,2048,1000,3,4,1,0,1000.000000
3,2048,1000,3,8,1,0,1000.000000
4,2048,1000,1,4,1,0,1000.000000
5,2048,1000,2,4,1,0,1000.000000
6,2048,1000,3,4,1,0,1000.000000
7,2048,1000,5,4,1,0,1000.000000


In [68]:
def run_experiment_with_theta(n, set_size, k, cap, theta):
    cbf = CountingBloomFilter(k=k, n=n, cap=cap)
    unique_strings = [generate_random_string(10) for _ in range(set_size)]
    false_negative_count = 0

    for s in unique_strings:
        cbf.put(s)

    for s in unique_strings:
        if not cbf.get(s, theta):
            false_negative_count += 1

    average_count = cbf.size()

    return {
        'n': n,
        'set_size': set_size,
        'k': k,
        'cap': cap,
        'theta': theta,
        'false_negative_count': false_negative_count,
        'average_count': average_count
    }

n = 2048
set_size = 1000
k = 3
cap = 4
theta_values = [1, 2, 3, 4]

results = []
for theta in theta_values:
    result = run_experiment_with_theta(n=n, set_size=set_size, k=k, cap=cap, theta=theta)
    results.append(result)

pd.DataFrame(results)

,n,set_size,k,cap,theta,false_negative_count,average_count
0,2048,1000,3,4,1,0,1000.0
1,2048,1000,3,4,2,562,1000.0
2,2048,1000,3,4,3,933,1000.0
3,2048,1000,3,4,4,997,1000.0


### Задание 4

Реализация HyperLogLog.

In [61]:
class HyperLogLog:
    def __init__(self, k, n):
        self.k = k
        self.n = n
        self.buckets = np.zeros(self.n, dtype=np.uint8)

    def _hash(self, s):
        hash_value = mmh3.hash(s, 42)
        return format(hash_value, '032b')

    def put(self, s):
        hash_bin = self._hash(s)
        bucket_index = int(hash_bin[:self.k], 2)
        remaining_bits = hash_bin[self.k:]
        leading_zeros = len(remaining_bits) - len(remaining_bits.lstrip('0'))
        self.buckets[bucket_index] = max(self.buckets[bucket_index], leading_zeros + 1)

    def est_size(self):
        Z = 0
        for bucket in self.buckets:
            Z += 2 ** -bucket 
        
        alpha_m = 0.7213 / (1 + 1.079 / self.n)
        raw_estimate = alpha_m * self.n ** 2 / Z
        
        if raw_estimate <= 2.5 * self.n:
            V = sum(1 for x in self.buckets if x == 0)
            if V > 0:
                raw_estimate = self.n * math.log(self.n / V)
        
        return int(raw_estimate)

* Проверим, как алгоритм ведет себя при малом количестве элементов.
* Проверим масштабируемость алгоритма и его точность при увеличении числа элементов.
* Добавим набор уникальных элементов, а затем повторим их несколько раз, чтобы убедиться, что HyperLogLog не учитывает повторяющиеся элементы и оценка количества уникальных элементов остаётся постоянной.
* Проведем эксперимент с различными значениями k, чтобы понять, как изменение количества бакетов влияет на точность алгоритма.
* Используем короткие строки, которые могут легко хешироваться в одно и то же значение, чтобы проверить устойчивость алгоритма к хеш-коллизиям.

In [64]:
def run_large_hll_experiment(k, n, num_elements):
    hll = HyperLogLog(k=k, n=n)
    unique_strings = {generate_random_string(10) for _ in range(num_elements)}
    start_time = time.time()
    
    for s in unique_strings:
        hll.put(s)
    
    estimated_size = hll.est_size()
    execution_time = time.time() - start_time

    return {
        'k': k,
        'n': n,
        'actual_size': len(unique_strings),
        'estimated_size': estimated_size,
        'error': abs(len(unique_strings) - estimated_size),
        'execution_time': execution_time
    }

k_values = list(range(10, 21))
element_counts = [10**6, 10**7]
results = []

for num_elements in element_counts:
    for k in k_values:
        n = 2**k
        result = run_large_hll_experiment(k, n, num_elements)
        results.append(result)

pd.DataFrame(results)

C:\Users\asmir\AppData\Local\Temp\ipykernel_4576\2998940640.py:21: RuntimeWarning: overflow encountered in scalar negative
  Z += 2 ** -bucket


     k        n  actual_size  estimated_size         error  execution_time
0   10     1024      1000000          755541        244459        5.669486
1   11     2048      1000000         3023758       2023758        5.717801
2   12     4096      1000000        12098218      11098218        6.862132
3   13     8192      1000000        48399248      47399248        6.169022
4   14    16384      1000000       193609743     192609743        5.809991
5   15    32768      1000000       774464475     773464475        5.908292
6   16    65536      1000000      3097908905    3096908905        6.049579
7   17   131072      1000000       210029451     209029451        6.394550
8   18   262144      1000000         8587518       7587518        7.138272
9   19   524288      1000000         2547822       1547822        8.285583
10  20  1048576      1000000         1001363          1363       12.741971
11  10     1024     10000000          755541       9244459       57.848625
12  11     2048     10000

* Точность HyperLogLog очень высока для небольших множеств (до 50 элементов) даже при малом количестве бакетов.
* Ошибка начинает расти при увеличении фактического количества элементов, особенно при малом количестве бакетов (n = 256), что вызывает завышенные оценки.
* Увеличение количества бакетов снижает ошибку, особенно для более крупных наборов данных.
* Количество хеш-функций (k) влияет на точность при работе с большими множествами. Меньшие значения k (k=8) при высоких значениях n дают более точные результаты.